### load patches

In [ ]:
import numpy as np, zarr, json, os
from pathlib import Path
from scipy import ndimage as ndi
from skimage.segmentation import watershed

TRAIN_ITER = "v2"
OVERWRITE = True

VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # used in baseline (v0)
    "6bba_61dd1e0d", # added in first iter (v1)

]
MAX_PATCHES = {
    "6bba_61dd1e0d": 100, # added in first iter (v1)
}

raw, lab = {}, {}
for p in VIDEOS:
    r = np.asarray(zarr.open(f'unet_patches/{p}.zarr/0', mode="r"))
    l = np.load(f'unet_labels/{p}_pts.npy', allow_pickle=True)
    n = MAX_PATCHES.get(p)
    if n is not None:
        r = r[:n]
        if p == "background":
            l = np.zeros((0, 4), dtype=np.int32) 
        else:
            l = l[l[:, 0] < n]                         
    raw[p], lab[p] = r, l
    print(p, raw[p].shape, lab[p].shape)

44b6_33b596bf (50, 16, 64, 64) (678, 4)
44b6_40c45f5a (200, 16, 64, 64) (3782, 4)
44b6_551a5dba (228, 16, 64, 64) (6196, 4)
6bba_61dd1e0d (100, 16, 64, 64) (231, 4)


### Gaussian heatmaps

In [ ]:
# modified gaussain maps in iter 2
FLOOR = np.array([1.0, 4.0, 4.0])     # never go below the current fixed sigma
CAP   = np.array([3.0, 12.0, 12.0])   # largest sigma allowed (z, y, x), must fit the patch
FRAC  = 0.35                          # fraction of the watershed radius used as sigma
MAX_DIST = 40                         # flood limit from a seed, in xy-voxel units
ZUP = 4                               # z is 4x coarser than xy, so upsample z by 4 to make voxels isotropic

zz, yy, xx = np.ogrid[:16, :64, :64]

def pts_for_patch(arr, i):
    if arr.dtype == object:                          # one array of points per patch
        return np.asarray(arr[i], float).reshape(-1, 3)
    if arr.ndim == 2 and arr.shape[1] == 4:          # rows of (patch_idx, z, y, x)
        return arr[arr[:, 0] == i, 1:].astype(float)
    raise ValueError(f"unrecognised points layout: {arr.shape}")

def sigmas_for_patch(img, pts):
    """Per-point (sz, sy, sx) from a seeded watershed. Falls back to FLOOR on any problem."""
    k = len(pts)
    sig = np.tile(FLOOR, (k, 1))
    if k == 0:
        return sig
    img = img.astype(np.float32)
    rng = img.max() - img.min()
    if rng < 1e-6:
        return sig
    im = (img - img.min()) / rng
    im = ndi.gaussian_filter(im, (0.5, 2, 2))
    im = ndi.zoom(im, (ZUP, 1, 1), order=1)                       # (64, 64, 64), ~isotropic

    markers = np.zeros(im.shape, np.int32)
    for j, (z, y, x) in enumerate(pts, 1):
        iz = int(np.clip(round(z * ZUP + (ZUP - 1) / 2), 0, im.shape[0] - 1))
        iy = int(np.clip(round(y), 0, 63)); ix = int(np.clip(round(x), 0, 63))
        markers[iz, iy, ix] = j                                    # duplicates overwrite -> fallback below

    # limit flooding to a ball around the seeds so a lone seed can't flood the patch
    dist = ndi.distance_transform_edt(markers == 0)
    mask = dist <= MAX_DIST
    lab_ = watershed(-im, markers, mask=mask, compactness=0.01)

    for j in range(1, k + 1):
        c = np.argwhere(lab_ == j)
        if len(c) < 20:                                            # empty or tiny region
            continue
        # uniform ellipsoid: std = r / sqrt(5)  ->  r = sqrt(5) * std (robust to thin leaks)
        r = np.sqrt(5.0) * c.std(0)
        r[0] /= ZUP                                                # back to original z voxels
        sig[j - 1] = np.clip(FRAC * r, FLOOR, CAP)
    return sig

def make_heatmap(pts, sigmas):
    hm = np.zeros((16, 64, 64), np.float32)
    for (cz, cy, cx), (sz, sy, sx) in zip(pts, sigmas):
        g = np.exp(-(((zz - cz) / sz) ** 2 + ((yy - cy) / sy) ** 2 + ((xx - cx) / sx) ** 2) / 2)
        hm = np.maximum(hm, g.astype(np.float32))
    return hm


In [ ]:
heat, all_sig = {}, {}
for name in VIDEOS:
    out = Path("heatmaps") / f"{name}_{TRAIN_ITER}_heatmap.npy"
    sig_out = Path("heatmaps") / f"{name}_{TRAIN_ITER}_sigmas.npy"

    if not OVERWRITE and out.exists():
        heat[name] = np.load(out); print("loaded existing:", out); continue

    out.parent.mkdir(parents=True, exist_ok=True)

    # background patches: no cells, so the target is all zeros (no watershed, no sigmas)
    if name == "background":
        heat[name] = np.zeros((raw[name].shape[0], 16, 64, 64), np.float32)
        np.save(out, heat[name])
        print(name, "generated", heat[name].shape, "| all-zero targets")
        continue

    R = raw[name].squeeze() if raw[name].ndim == 5 else raw[name]
    N = R.shape[0]
    hms, sgs = [], []
    for i in range(N):
        pts = pts_for_patch(lab[name], i)
        s = sigmas_for_patch(R[i], pts)
        hms.append(make_heatmap(pts, s)); sgs.append(s)
    heat[name] = np.stack(hms)
    all_sig[name] = np.concatenate(sgs) if sgs else np.zeros((0, 3))
    np.save(out, heat[name]); np.save(sig_out, all_sig[name])

    s = all_sig[name]
    print(name, "generated", heat[name].shape,
          "| patches with no points:", int((heat[name].reshape(N, -1).max(1) == 0).sum()))
    print("   sigma median (z,y,x):", np.median(s, 0).round(2),
          "| 95th pct:", np.percentile(s, 95, axis=0).round(2),
          "| at floor:", f"{(np.all(s <= FLOOR + 1e-6, axis=1)).mean():.0%}",
          "| at cap:", f"{(np.any(s >= CAP - 1e-6, axis=1)).mean():.0%}")

44b6_33b596bf generated (50, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.12 4.   4.  ] | 95th pct: [2.29 4.   4.  ] | at floor: 36% | at cap: 1%
44b6_40c45f5a generated (200, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.24 4.   4.  ] | 95th pct: [2.32 4.34 4.26] | at floor: 30% | at cap: 1%
44b6_551a5dba generated (228, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.03 4.   4.  ] | 95th pct: [2.4  4.   4.59] | at floor: 46% | at cap: 2%
6bba_61dd1e0d generated (100, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.34 4.   4.  ] | 95th pct: [2.57 5.23 6.1 ] | at floor: 34% | at cap: 0%


### preprocessing

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

norm_path = f"models/norm_{TRAIN_ITER}.json"

if not OVERWRITE and os.path.exists(norm_path):
    with open(norm_path, "r") as f:
        norm_params = json.load(f)
    lo, hi = norm_params["lo"], norm_params["hi"]
    print("loaded existing norm:", norm_path)
else:
    allraw = np.concatenate([raw[n].ravel() for n in VIDEOS])
    lo, hi = np.percentile(allraw, [1, 99.8])

    with open(norm_path, "w") as f:
        json.dump({"lo": float(lo), "hi": float(hi)}, f)

    print("computed and saved norm:", norm_path)

print("norm range:", lo, hi)

def norm(x):
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

X = np.concatenate([norm(raw[n]) for n in VIDEOS])[:, None]
Y = np.concatenate([heat[n] for n in VIDEOS])[:, None]

print(X.shape, Y.shape, "target max:", Y.max())

loader = DataLoader(
    TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
    batch_size=8,
    shuffle=True,
    drop_last=False
)

computed and saved norm: models/norm_v3.json
norm range: 0.0 3553.0
(728, 1, 16, 64, 64) (728, 1, 16, 64, 64) target max: 1.0


### 3D U-Net

In [7]:
%%writefile unet3d.py
import torch, torch.nn as nn

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True),
        nn.Conv3d(o, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True)
        )

class UNet3D(nn.Module):
    # anisotropic pooling: z is only 16 slices, so don't squash it early
    def __init__(self, c=16):
        super().__init__()
        self.e1, self.e2, self.e3, self.b = block(1, c), block(c, 2*c), block(2*c, 4*c), block(4*c, 8*c)
        self.p1, self.p2, self.p3 = nn.MaxPool3d((1,2,2)), nn.MaxPool3d(2), nn.MaxPool3d(2)
        self.u3 = nn.ConvTranspose3d(8*c, 4*c, 2, stride=2)
        self.u2 = nn.ConvTranspose3d(4*c, 2*c, 2, stride=2)
        self.u1 = nn.ConvTranspose3d(2*c, c, (1,2,2), stride=(1,2,2))
        self.d3, self.d2, self.d1 = block(8*c, 4*c), block(4*c, 2*c), block(2*c, c)
        self.out = nn.Conv3d(c, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

Overwriting unet3d.py


In [8]:
from unet3d import UNet3D
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet3D(16).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(device, sum(p.numel() for p in model.parameters())/1e6, "M params")

cuda 1.399921 M params


### training

In [ ]:
EPOCHS, SAVE_EVERY = 100, 10

def weighted_mse(pred, target):
    w = 1 + 20 * target          # upweight cell voxels, background is the vast majority
    return (w * (pred - target) ** 2).mean()

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)   
        opt.zero_grad()
        loss = weighted_mse(model(xb), yb)
        loss.backward(); opt.step()
        tot += loss.item() * len(xb)
    print(f"epoch {ep:3d}  loss {tot/len(X):.5f}")
    if ep % SAVE_EVERY == 0:
        torch.save(model.state_dict(), f"models/unet_{TRAIN_ITER}_e{ep}.pt")

epoch   1  loss 0.13975
epoch   2  loss 0.07892
epoch   3  loss 0.06389
epoch   4  loss 0.05796
epoch   5  loss 0.05049
epoch   6  loss 0.04476
epoch   7  loss 0.04171
epoch   8  loss 0.03695
epoch   9  loss 0.03402
epoch  10  loss 0.03174
epoch  11  loss 0.03202
epoch  12  loss 0.02861
epoch  13  loss 0.02654
epoch  14  loss 0.02562
epoch  15  loss 0.02344
epoch  16  loss 0.02171
epoch  17  loss 0.01992
epoch  18  loss 0.01852
epoch  19  loss 0.01746
epoch  20  loss 0.01760
epoch  21  loss 0.01646
epoch  22  loss 0.01465
epoch  23  loss 0.01362
epoch  24  loss 0.01454
epoch  25  loss 0.01242
epoch  26  loss 0.01097
epoch  27  loss 0.01110
epoch  28  loss 0.01014
epoch  29  loss 0.01098
epoch  30  loss 0.01087
epoch  31  loss 0.00911
epoch  32  loss 0.00817
epoch  33  loss 0.00833
epoch  34  loss 0.00724
epoch  35  loss 0.00761
epoch  36  loss 0.00868
epoch  37  loss 0.00658
epoch  38  loss 0.00680
epoch  39  loss 0.00575
epoch  40  loss 0.00499
epoch  41  loss 0.00578
epoch  42  loss 

: 